## Explode:

- explode() is a PySpark function is used to `converts each element of an array into a separate row in a DataFrame`.

- **Simple explanation:** One row contains `multiple values`. explode() splits `those values into separate rows`.

### Before Exploding:

| OrderID | Products                 |
|---------|--------------------------|
| 101     | ["Laptop", "Mouse"]      |
| 102     | ["Keyboard", "Monitor"]  |

### After Exploding:

| OrderID | Product  |
|---------|----------|
| 101     | Laptop   |
| 101     | Mouse    |
| 102     | Keyboard |
| 102     | Monitor  |

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode

spark = SparkSession.builder.appName("Explode").getOrCreate()

orders_df = spark.createDataFrame([
    (101, ["Laptop", "Mouse"]),
    (102, ["Keyboard", "Monitor"]),
    (103, ["Mobile", "Charger"])
], ["OrderID", "Products"])

exploded_df = orders_df.withColumn(
    "Product",
    explode("Products")
)

exploded_df.select("OrderID", "Product").show()

### An E-commerce company stores multiple products in a single order. The data engineering team needs one row per product to calculate product-wise sales and analyze orders.

%md
### Input Data:

| OrderID | Customer | Products                  |
|---------|----------|---------------------------|
| 1001    | Arun     | Laptop, Mouse             |
| 1002    | Bala     | Keyboard, Mouse, USB       |
| 1003    | Priya    | Monitor                   |

### Output: One Row Per Product

| OrderID | Customer | Product  |
|---------|----------|----------|
| 1001    | Arun     | Laptop   |
| 1001    | Arun     | Mouse    |
| 1002    | Bala     | Keyboard |
| 1002    | Bala     | Mouse    |
| 1002    | Bala     | USB      |
| 1003    | Priya    | Monitor  |

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode

spark = SparkSession.builder.appName(
    "EcommerceExplode"
).getOrCreate()

orders_df = spark.createDataFrame([
    (1001, "Arun", ["Laptop", "Mouse"]),
    (1002, "Bala", ["Keyboard", "Mouse", "USB"]),
    (1003, "Priya", ["Monitor"])
], ["OrderID", "Customer", "Products"])

print("Original orders:")
orders_df.show(truncate=False)

# Convert each product into a separate row
product_df = orders_df.select(
    "OrderID",
    "Customer",
    explode("Products").alias("Product")
)

print("Exploded products:")
product_df.show()


### The company wants to know how many times each product appears (Count product orders).

In [0]:
from pyspark.sql.functions import *

product_df.groupBy("Product").agg(
    count("*").alias("product_count")
).show()